In [1]:
import xarray as xr
import pandas as pd
from scipy.stats import linregress
import matplotlib.pyplot as plt

In [2]:
def classify_precip(row):
    code = row['WEATHER']
    date = row['DATE']
    year = date.year  # extract year from datetime

    if code == 5:
        return 'Non-Precipitation'
    elif code in [6, 7]:
        return 'Non-Precipitation' if year >= 2007 else 'Precipitation'
    else:
        return 'Precipitation'

In [3]:
def detrend_state(df):
    df = df.sort_values('DATE')
    x = df['DATE'].map(pd.Timestamp.toordinal)
    y = df['CRASH_COUNTS']
    slope, intercept, r_value, p_value, std_err = linregress(x, y)
    df['DETRENDED_CRASH_COUNTS'] = y - (slope*x + intercept)
    return df


In [4]:
def read_psl_standard(filename, missing_value_marker=-999):
    """
    Read a NOAA PSL "standard format" time-series file into a pandas DataFrame.

    This function parses the PSL standard format:
      - First line contains start and end years
      - Each subsequent line has a year followed by 12 monthly values
      - A sentinel line with a missing-value marker (e.g., -999) follows the last year
      - Any additional lines (metadata/info) are ignored

    Returns
    -------
    df : pandas.DataFrame
        Wide format with columns: ['year', 'month_1', ..., 'month_12'].
    df_long : pandas.DataFrame
        Long format with columns: ['year', 'month', 'value'].

    Note
    ----
    This code was generated with the assistance of ChatGPT.
    """

    with open(filename, 'r') as f:
        # First line: contains the start and end year (not used directly, but can be checked)
        first_line = f.readline().strip().split()
        year_start, year_end = map(int, first_line)

        data_lines = []
        while True:
            line = f.readline()
            if not line:
                # Safety: file ended before sentinel found
                raise ValueError("Reached EOF before missing-value marker")
            parts = line.strip().split()
            # Stop when we reach the sentinel missing-value marker (e.g., -999)
            if len(parts) == 1 and parts[0] == str(missing_value_marker):
                break
            data_lines.append(parts)

        # Create DataFrame with year + 12 monthly values
        df = pd.DataFrame(data_lines, columns=['YEAR'] + [f'MONTH_{i}' for i in range(1, 13)])
        # Convert year to int and monthly values to float
        df = df.astype(float)
        df['YEAR'] = df['YEAR'].astype(int)

        # Reshape into long format for easier time-series analysis
        df_long = df.melt(id_vars='YEAR', var_name='MONTH', value_name='SST_VALUE')
        # Convert 'month' from "month_1" → integer 1
        df_long['MONTH'] = df_long['MONTH'].str.replace('MONTH_', '').astype(int)

        return df, df_long

In [5]:
state_dictAB = {
    1: 'AL', 2: 'AK', 4: 'AZ', 5: 'AR', 6: 'CA', 8: 'CO', 9: 'CT', 10: 'DE',
    11: 'DC', 12: 'FL', 13: 'GA', 15: 'HI', 16: 'ID', 17: 'IL', 18: 'IN',
    19: 'IA', 20: 'KS', 21: 'KY', 22: 'LA', 23: 'ME', 24: 'MD', 25: 'MA',
    26: 'MI', 27: 'MN', 28: 'MS', 29: 'MO', 30: 'MT', 31: 'NE', 32: 'NV',
    33: 'NH', 34: 'NJ', 35: 'NM', 36: 'NY', 37: 'NC', 38: 'ND', 39: 'OH',
    40: 'OK', 41: 'OR', 42: 'PA', 44: 'RI', 45: 'SC', 46: 'SD', 47: 'TN',
    48: 'TX', 49: 'UT', 50: 'VT', 51: 'VA', 53: 'WA', 54: 'WV', 55: 'WI',
    56: 'WY'
}

In [6]:
excluded_states = {'DC', 'AK', 'HI', 'PR', 'PRCP'}

In [7]:
cluster_names = {
    0: "Alaskan Ridge/Pacific Ridge",
    1: "Greenland High",
    2: "Pacific Trough",
    3: "West Coast High"
}

# 1. Define FARS, NINO34, and PRECIP Data Files

In [8]:
fars_file='../data/FARS/old/DATABASE3.csv'
wx_regimes_file='../data/precip/state_precip_wxregimes.csv'
precip_file='../data/precip/state_precip_chirps.csv'

# Maps day to specific cluster
cluster_file='../data/wxregimes/kmeans_4cluster_DJF_1981-2019_NA.nc'

df_fars=pd.read_csv(fars_file)

# map states
df_fars['STATE_ABBR'] = df_fars['STATE'].map(state_dictAB)

# remove excluded states
df_fars=df_fars[~df_fars["STATE_ABBR"].isin(excluded_states)]

# Subset Years in fars
df_fars = df_fars[(df_fars["YEAR"] >= 1981) & (df_fars["YEAR"] <= 2019)]

# Build datetime column in df_fars
df_fars["DATE"] = pd.to_datetime(
    df_fars["YEAR"].astype(str).str.zfill(4)
    + df_fars["MONTH"].astype(str).str.zfill(2)
    + df_fars["DAY"].astype(str).str.zfill(2),
    format="%Y%m%d",
    errors="coerce"  # handles bad dates automatically
)

# Ensure DATE is datetime
df_fars["DATE"] = pd.to_datetime(df_fars["DATE"])

# Apply classification function
df_fars["precip_class"] = df_fars.apply(classify_precip, axis=1)

# Keep only precipitation-classified crash events
df_fars_precip = df_fars[df_fars["precip_class"] == "Precipitation"].copy()

# Remove the helper column if no longer needed
df_fars_precip.drop(columns=["precip_class"], inplace=True)

df_fars_precip

# 2. Read in Wx Regimes and FARS data & merge

In [9]:
# Load data
df_cluster = xr.open_dataset(cluster_file).to_dataframe().reset_index()

In [10]:
df_cluster

,time,cluster
0,1981-01-01,3
1,1981-01-02,3
2,1981-01-03,3
3,1981-01-04,3
4,1981-01-05,3
...,...,...
4723,2019-12-27,2
4724,2019-12-28,1
4725,2019-12-29,1
4726,2019-12-30,1


# Daily aggregation
df_daily = (
    df_fars_precip
    .groupby(["DATE", "STATE_ABBR"], as_index=False)["FATALS"]
    .sum()
    .rename(columns={"FATALS": "CRASH_COUNT"})
)

df_daily

In [11]:
# Reset index so 'time' becomes a column
df_cluster = df_cluster.reset_index(drop=True)

# Rename the index column to 'DATE' to match df_fars
df_cluster = df_cluster.rename(columns={"time": "DATE"})

# Ensure datetime
df_cluster["DATE"] = pd.to_datetime(df_cluster["DATE"])

In [12]:
df_cluster

,DATE,cluster
0,1981-01-01,3
1,1981-01-02,3
2,1981-01-03,3
3,1981-01-04,3
4,1981-01-05,3
...,...,...
4723,2019-12-27,2
4724,2019-12-28,1
4725,2019-12-29,1
4726,2019-12-30,1


# Merge on DATE
df_merged = pd.merge(df_daily, df_cluster, on="DATE", how="left")

# Add cluster names
df_merged["cluster_name"] = df_merged["cluster"].map(cluster_names)

df_merged

# 3. Add the Precip Anomalies and Sum for each state and date based on CHIRPS

In [13]:
df_state_precip=pd.read_csv(precip_file)

In [14]:
df_state_precip["DATE"] = pd.to_datetime(df_state_precip["DATE"])

In [15]:
# Remove excluded states
df_filtered = df_state_precip[~df_state_precip["STATE_ABBR"].isin(excluded_states)]
df_filtered = df_filtered.drop(columns=["STATE", "STATEFP"])
df_filtered

,DATE,ANOM_MEAN,ANOM_SUM,STATE_ABBR
0,1981-01-01,-4.391150,-917.750447,AL
1,1981-01-02,-4.417280,-923.211495,AL
2,1981-01-03,-4.426943,-925.231093,AL
3,1981-01-04,-4.419716,-923.720710,AL
4,1981-01-05,-4.411256,-921.952587,AL
...,...,...,...,...
172426,2019-12-27,-0.507263,-227.253931,WY
172427,2019-12-28,-0.266899,-119.570896,WY
172428,2019-12-29,-0.526915,-236.057830,WY
172429,2019-12-30,-0.526406,-235.829965,WY


In [25]:
# Ensure DATE columns are datetime
df_filtered['DATE'] = pd.to_datetime(df_filtered['DATE'])
df_cluster['DATE'] = pd.to_datetime(df_cluster['DATE'])

# Merge cluster assignment into filtered state anomalies
df_final = df_filtered.merge(
    df_cluster,
    left_on='DATE',
    right_on='DATE',
    how='left'
)


# Compute state × cluster composite anomalies
df_composite = (
    df_final
    .groupby(['STATE_ABBR', 'cluster'], as_index=False)
    .agg({'ANOM_MEAN': 'mean'})
    .rename(columns={'ANOM_MEAN': 'ANOM_MEAN_composite'})
)

# Inspect result
print(df_composite)

    STATE_ABBR  cluster  ANOM_MEAN_composite
0           AL        0             1.161320
1           AL        1             0.253434
2           AL        2            -0.160282
3           AL        3            -1.307853
4           AR        0             1.639292
..         ...      ...                  ...
187         WV        3            -1.099275
188         WY        0             0.083645
189         WY        1            -0.057930
190         WY        2             0.071458
191         WY        3            -0.211484

[192 rows x 3 columns]


# Merge precip anomalies into the merged dataset
df_final = pd.merge(
    df_merged,
    df_filtered[["DATE", "STATE_ABBR", "ANOM_MEAN", "ANOM_SUM"]],
    on=["DATE", "STATE_ABBR"],
    how="left"
)

In [73]:
outFile='../data/combined_databases/database_fars_wxregimes_precip_daily.csv'
df_final.to_csv(outFile)

# Compute state × regime composites over all days
df_composite = (
    df_final
    .groupby(["STATE_ABBR", "cluster_name"], as_index=False)
    .agg({
        "ANOM_MEAN": "mean",          # mean daily anomaly over all days in the regime
        "CRASH_COUNT": ["mean", "sum"]  # optional: mean daily crashes and total crashes
    })
)
df_composite

In [81]:
cluster_name = 'West Coast High'
df_cluster = df_final[df_final['cluster_name'] == cluster_name]
df_composite = (
    df_cluster
    .groupby('STATE_ABBR', as_index=False)
    .agg({
        "ANOM_MEAN": "mean",
        "CRASH_COUNT": ["mean", "sum"]
    })
)

# Flatten MultiIndex columns
df_composite.columns = [
    "STATE_ABBR", 
    "cluster_name", 
    "ANOM_MEAN_composite", 
    "CRASH_COUNT_mean_daily", 
    "CRASH_COUNT_total"
]

In [82]:
df_composite

STATE_ABBR ANOM_MEAN CRASH_COUNT      
                   mean        mean   sum
0          AL  5.008133    1.897727   334
1          AR  3.021908    1.677165   213
2          AZ  3.137165    1.544118   105
3          CA  2.839864    3.059028   881
4          CO  0.460239    1.556701   151
5          CT  5.351938    1.259740    97
6          DE  1.242412    1.250000    45
7          FL  3.482130    2.261628   778
8          GA  5.170565    2.075314   496
9          IA  0.063341    1.615385   210
10         ID  0.747393    1.041667    50
11         IL -0.363874    2.026490   612
12         IN -0.090585    1.869091   514
13         KS  0.876519    1.552239   104
14         KY  1.688305    1.657407   358
15         LA  5.638665    1.647727   290
16         MA  2.937233    1.320000   198
17         MD  1.560106    1.529851   205
18         ME  3.186161    1.208955    81
19         MI -0.288006    1.975359   962
20         MN -0.031170    1.557377   285
21         MO  1.270145    1.965517   342
22         MS  5.198000    1.675824   305
23         MT  0.349509    1.266667    57
24         NC  3.111826    1.993485   612
25         ND  0.062133    1.176471    40
26         NE  0.316132    1.344262    82
27         NH  3.007625    1.232143    69
28         NJ  2.548731    1.639594   323
29         NM  0.970186    1.267857    71
30         NV  0.480123    1.352941    46
31         NY  0.077354    1.970917   881
32         OH -0.326195    2.000000   804
33         OK  1.532382    1.567010   152
34         OR  1.741285    1.335766   183
35         PA  0.041710    2.070776   907
36         RI  8.006205    1.000000    30
37         SC  5.526823    1.764706   300
38         SD -0.070448    1.181818    52
39         TN  2.637050    1.783898   421
40         TX  1.137290    2.877660  1082
41         UT  0.482139    1.400000    63
42         VA  2.520358    1.607527   299
43         VT -0.190475    1.301887    69
44         WA  1.247110    1.412698   267
45         WI  0.252669    1.764103   344
46         WV  0.846592    1.402174   258
47         WY -0.002748    1.375000    44

In [79]:
outFile='../data/combined_databases/database_fars_wxregimes_precip_daily_summary_state.csv'
df_composite.to_csv(outFile)

In [80]:
df_composite[df_composite['cluster_name'] == 'West Coast High']

,STATE_ABBR,cluster_name,ANOM_MEAN_composite,CRASH_COUNT_mean_daily,CRASH_COUNT_total
3,AL,West Coast High,5.008133,1.897727,334
7,AR,West Coast High,3.021908,1.677165,213
11,AZ,West Coast High,3.137165,1.544118,105
15,CA,West Coast High,2.839864,3.059028,881
19,CO,West Coast High,0.460239,1.556701,151
23,CT,West Coast High,5.351938,1.259740,97
27,DE,West Coast High,1.242412,1.250000,45
31,FL,West Coast High,3.482130,2.261628,778
35,GA,West Coast High,5.170565,2.075314,496
39,IA,West Coast High,0.063341,1.615385,210


outFile='../data/combined_databases/database_fars_wxregimes_precip_daily_summary_state.csv'
df_state_cluster.to_csv(outFile)

# 1️⃣ Aggregate daily totals nationwide
daily_national = (
    df_combined.groupby('DATE', as_index=False)
    .agg({'CRASH_COUNTS': 'sum'})
)

# 2️⃣ Extract YEAR, MONTH, DAY for convenience
daily_national['YEAR'] = daily_national['DATE'].dt.year
daily_national['MONTH'] = daily_national['DATE'].dt.month
daily_national['DAY'] = daily_national['DATE'].dt.day

# 3️⃣ Compute daily-of-month anomaly (seasonal anomaly)
daily_national['ANOM_CRASH_COUNTS'] = daily_national['CRASH_COUNTS'] - \
    daily_national.groupby(['MONTH', 'DAY'])['CRASH_COUNTS'].transform('mean')

# 4️⃣ Detrend national crash counts
x = daily_national['DATE'].map(pd.Timestamp.toordinal)
y = daily_national['CRASH_COUNTS']
slope, intercept, r_value, p_value, std_err = linregress(x, y)
daily_national['DETRENDED_CRASH_COUNTS'] = y - (slope*x + intercept)

# 5️⃣ Add pseudo-state column for consistency
daily_national['STATE_ABBR'] = 'USA'

# 6️⃣ Optional: reorder columns
cols_order = [
    'STATE_ABBR', 'YEAR', 'MONTH', 'DAY',
    'CRASH_COUNTS', 'ANOM_CRASH_COUNTS', 'DETRENDED_CRASH_COUNTS', 'DATE'
]
daily_national = daily_national[cols_order]

# ✅ Inspect
daily_national.head()

# Select USA data
usa = daily_national[daily_national['STATE_ABBR'] == 'USA'].sort_values('DATE')

plt.figure(figsize=(12,5))
plt.plot(usa['DATE'], usa['ANOM_CRASH_COUNTS'], color='black', lw=1.5, label='Anomaly')
plt.plot(usa['DATE'],daily_national.groupby(['MONTH','DAY'])['CRASH_COUNTS'].transform('mean'),label='Climo')
plt.axhline(0, color='gray', lw=1, linestyle='--')
plt.xlabel('Date')
plt.ylabel('Crash Anomaly')
plt.title('Daily Fatal Crash Anomalies - USA')
plt.grid(axis='y', linestyle=':', alpha=0.4)
#plt.legend()
plt.tight_layout()
plt.show()

# Choose states to plot
states_to_plot = ['CA', 'TX', 'FL', 'NY', 'IL']  # example top 5 states

# Prepare data
df_states_plot = df_daily_state[df_daily_state['STATE_ABBR'].isin(states_to_plot)].copy()

# Ensure DATE is datetime
df_states_plot['DATE'] = pd.to_datetime(df_states_plot['DATE'])

# Plot
plt.figure(figsize=(16,6))

for state in states_to_plot:
    df_s = df_states_plot[df_states_plot['STATE_ABBR'] == state]
    
    # Plot seasonal anomaly
    plt.plot(df_s['DATE'], df_s['ANOM_CRASH_COUNTS'], 
             label=f"{state} anomaly", linewidth=1.5, alpha=0.7)
    
    # Plot state mean as background line (optional)
    state_mean = df_s['CRASH_COUNTS'].mean()
    plt.axhline(state_mean, color='gray', linestyle='--', alpha=0.3)
    
plt.axhline(0, color='black', linestyle='-', linewidth=1)
plt.title('Daily Fatal Crash Anomalies with State Means')
plt.ylabel('Crash Anomaly')
plt.xlabel('Date')
plt.xticks(rotation=45)
plt.legend(fontsize=12)
plt.tight_layout()
plt.show()


print("raw dtype:", df_combined['PRECIP_ANOM_MEAN'].dtype)
print("raw head:")
print(df_combined['PRECIP_ANOM_MEAN'].head(10))
print("describe (raw):")
print(df_combined['PRECIP_ANOM_MEAN'].describe())

# count negatives in the raw combined table
raw_neg = (pd.to_numeric(df_combined['PRECIP_ANOM_MEAN'], errors='coerce') < 0).sum()
print("raw negative count:", raw_neg)

# If you have a previously computed df_daily_state, check it too:
if 'df_daily_state' in globals():
    print("daily aggregated describe:")
    print(df_daily_state['PRECIP_ANOM_MEAN'].describe())
    print("daily negative count:", (pd.to_numeric(df_daily_state['PRECIP_ANOM_MEAN'], errors='coerce') < 0).sum())
else:
    print("df_daily_state not found — will compute fresh below.")